In [ ]:
"""
CREDIT RISK ASSESSMENT - MODELING
Run this entire cell for complete model training and evaluation
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE
import warnings
import joblib
warnings.filterwarnings('ignore')

print("=" * 70)
print("CREDIT RISK ASSESSMENT - MODELING")
print("=" * 70)

# ============================================
# LOAD FEATURED DATA
# ============================================

print("\n Loading featured data...")
df = pd.read_csv('C:/Users/AlexB/Desktop/credit_risk_assessment/data/processed/credit_data_featured.csv')
print(f" Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

# ============================================
# PREPARE DATA FOR MODELING
# ============================================

print("\n" + "=" * 70)
print("PREPARING DATA FOR MODELING")
print("=" * 70)

# Exclude categorical group columns (created during EDA)
exclude_cols = ['SeriousDlqin2yrs', 'AgeGroup', 'UtilGroup', 'IncomeGroup', 'DebtGroup']
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols]
y = df['SeriousDlqin2yrs']

print(f" Features: {X.shape[1]} columns")
print(f" Target: {y.name}")
print(f" Default rate: {y.mean():.2%}")

# ============================================
# TRAIN-TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n Train set: {X_train.shape[0]:,} rows")
print(f" Test set: {X_test.shape[0]:,} rows")
print(f" Train default rate: {y_train.mean():.2%}")
print(f" Test default rate: {y_test.mean():.2%}")

# ============================================
# FEATURE SCALING
# ============================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n Features scaled using StandardScaler")

# ============================================
# APPLY SMOTE FOR IMBALANCED DATA
# ============================================

print("\n" + "=" * 70)
print("APPLYING SMOTE FOR CLASS IMBALANCE")
print("=" * 70)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f" Before SMOTE: {len(y_train):,} rows")
print(f" After SMOTE: {len(y_train_balanced):,} rows")
print(f" Default rate after SMOTE: {y_train_balanced.mean():.2%}")

# ============================================
# TRAIN MODELS
# ============================================

print("\n" + "=" * 70)
print("TRAINING MODELS")
print("=" * 70)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

results = {}

for name, model in models.items():
    print(f"\n Training {name}...")
    model.fit(X_train_balanced, y_train_balanced)
    
    # Predict on test set
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'auc': auc
    }
    print(f"    Test AUC: {auc:.4f}")

# ============================================
# MODEL COMPARISON
# ============================================

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

comparison = pd.DataFrame([(name, results[name]['auc']) for name in results.keys()], 
                           columns=['Model', 'Test AUC'])
comparison = comparison.sort_values('Test AUC', ascending=False)

print("\n Model Performance:")
print(comparison.to_string(index=False))

# Best model
best_model_name = comparison.iloc[0]['Model']
best_model = results[best_model_name]['model']
best_auc = comparison.iloc[0]['Test AUC']

print(f"\n🏆 Best Model: {best_model_name} (AUC: {best_auc:.4f})")

# ============================================
# HYPERPARAMETER TUNING FOR BEST MODEL
# ============================================

print("\n" + "=" * 70)
print(f"HYPERPARAMETER TUNING - {best_model_name}")
print("=" * 70)

if best_model_name == 'XGBoost':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 1.0]
    }
    base_model = XGBClassifier(random_state=42, eval_metric='logloss')
    
elif best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5, 10]
    }
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
    
elif best_model_name == 'Gradient Boosting':
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7]
    }
    base_model = GradientBoostingClassifier(random_state=42)
    
else:  # Logistic Regression
    param_grid = {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2']
    }
    base_model = LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

print(f" Tuning {best_model_name}...")
print(f"   Parameter grid: {param_grid}")

grid_search = GridSearchCV(base_model, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_search.fit(X_train_balanced, y_train_balanced)

print(f"\n Best parameters: {grid_search.best_params_}")
print(f" Best CV AUC: {grid_search.best_score_:.4f}")

# Evaluate tuned model
tuned_model = grid_search.best_estimator_
y_pred_proba_tuned = tuned_model.predict_proba(X_test_scaled)[:, 1]
tuned_auc = roc_auc_score(y_test, y_pred_proba_tuned)

print(f" Tuned model test AUC: {tuned_auc:.4f}")

# ============================================
# FINAL MODEL EVALUATION
# ============================================

print("\n" + "=" * 70)
print("FINAL MODEL EVALUATION")
print("=" * 70)

# Find optimal threshold (Youden's J statistic)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_tuned)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f" Optimal threshold: {optimal_threshold:.4f}")

# Predictions at optimal threshold
y_pred_optimal = (y_pred_proba_tuned >= optimal_threshold).astype(int)

# Classification Report
print("\n Classification Report:")
print(classification_report(y_test, y_pred_optimal, target_names=['Good Credit', 'Default']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_optimal)
print("\n Confusion Matrix:")
print(f"   True Negatives (Good correctly identified): {cm[0,0]:,}")
print(f"   False Positives (Good incorrectly as Default): {cm[0,1]:,}")
print(f"   False Negatives (Default incorrectly as Good): {cm[1,0]:,}")
print(f"   True Positives (Default correctly identified): {cm[1,1]:,}")

# Calculate metrics
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n Key Metrics:")
print(f"   Precision: {precision:.4f} (Of predicted defaults, {precision:.1%} were correct)")
print(f"   Recall: {recall:.4f} (Found {recall:.1%} of actual defaults)")
print(f"   Specificity: {specificity:.4f} (Correctly identified {specificity:.1%} of good customers)")
print(f"   F1 Score: {f1:.4f}")

# ============================================
# FEATURE IMPORTANCE (for tree-based models)
# ============================================

if hasattr(tuned_model, 'feature_importances_'):
    print("\n" + "=" * 70)
    print("FEATURE IMPORTANCE")
    print("=" * 70)
    
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': tuned_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n Top 15 Most Important Features:")
    for i, row in feature_importance.head(15).iterrows():
        bar = "█" * int(row['Importance'] * 50)
        print(f"   {row['Feature']:35} {row['Importance']:.4f} {bar}")

# ============================================
# SAVE MODEL AND ARTIFACTS
# ============================================

print("\n" + "=" * 70)
print("SAVING MODEL AND ARTIFACTS")
print("=" * 70)

import os
os.makedirs('C:/Users/AlexB/Desktop/credit_risk_assessment/models/saved_models', exist_ok=True)

# Save model
model_path = 'C:/Users/AlexB/Desktop/credit_risk_assessment/models/saved_models/best_model.pkl'
joblib.dump(tuned_model, model_path)
print(f" Model saved to: {model_path}")

# Save scaler
scaler_path = 'C:/Users/AlexB/Desktop/credit_risk_assessment/models/saved_models/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f" Scaler saved to: {scaler_path}")

# Save feature list
features_path = 'C:/Users/AlexB/Desktop/credit_risk_assessment/models/saved_models/feature_columns.pkl'
joblib.dump(feature_cols, features_path)
print(f" Feature list saved to: {features_path}")

# Save threshold
threshold_path = 'C:/Users/AlexB/Desktop/credit_risk_assessment/models/saved_models/optimal_threshold.pkl'
joblib.dump(optimal_threshold, threshold_path)
print(f" Optimal threshold saved to: {threshold_path}")

# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "=" * 70)
print("🎉 PROJECT COMPLETE! 🎉")
print("=" * 70)

print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    CREDIT RISK ASSESSMENT - FINAL SUMMARY                    ║
╚══════════════════════════════════════════════════════════════════════════════╝

 DATA SUMMARY:
   ├── Original data: 150,000 rows, 11 columns
   ├── Cleaned data: 149,391 rows
   ├── Features engineered: {len(feature_cols)} columns
   └── Default rate: {y.mean():.2%}

MODEL PERFORMANCE:
   ├── Best Model: {best_model_name}
   ├── Test AUC: {tuned_auc:.4f}
   ├── Optimal Threshold: {optimal_threshold:.4f}
   ├── Precision: {precision:.4f}
   ├── Recall: {recall:.4f}
   ├── Specificity: {specificity:.4f}
   └── F1 Score: {f1:.4f}

 SAVED FILES:
   ├── data/processed/credit_data_cleaned.csv
   ├── data/processed/credit_data_featured.csv
   ├── models/saved_models/best_model.pkl
   ├── models/saved_models/scaler.pkl
   ├── models/saved_models/feature_columns.pkl
   └── models/saved_models/optimal_threshold.pkl

 BUSINESS INSIGHTS:
   1. Severe delinquency (90+ days) is the strongest predictor
   2. Credit utilization >80% significantly increases default risk
   3. Past delinquency history is a key indicator
   4. Model can identify {recall:.1%} of potential defaults
   5. {specificity:.1%} of good customers correctly identified

 PROJECT COMPLETED SUCCESSFULLY!
""")

print("=" * 70)

CREDIT RISK ASSESSMENT - MODELING

 Loading featured data...
 Loaded: 149,391 rows, 24 columns

PREPARING DATA FOR MODELING
 Features: 23 columns
 Target: SeriousDlqin2yrs
 Default rate: 6.70%

 Train set: 104,573 rows
 Test set: 44,818 rows
 Train default rate: 6.70%
 Test default rate: 6.70%

 Features scaled using StandardScaler

APPLYING SMOTE FOR CLASS IMBALANCE
 Before SMOTE: 104,573 rows
 After SMOTE: 195,134 rows
 Default rate after SMOTE: 50.00%

TRAINING MODELS

 Training Logistic Regression...
    Test AUC: 0.8604

 Training Random Forest...
    Test AUC: 0.8321

 Training Gradient Boosting...
    Test AUC: 0.8605

 Training XGBoost...
    Test AUC: 0.8451

MODEL COMPARISON

 Model Performance:
              Model  Test AUC
  Gradient Boosting  0.860530
Logistic Regression  0.860385
            XGBoost  0.845064
      Random Forest  0.832118

🏆 Best Model: Gradient Boosting (AUC: 0.8605)

HYPERPARAMETER TUNING - Gradient Boosting
 Tuning Gradient Boosting...
   Parameter gri